In [5]:
import numpy as np 
import pandas as pd 

from sklearn.preprocessing import MinMaxScaler, StandardScaler

import warnings
warnings.simplefilter(action = "ignore", category = RuntimeWarning)

input_file_path = '../input'
output_file_path = '../output'

# Data Pre-processing

In [ ]:
def combine_data(mens_data, womens_data):
    """
    This method combines individual mens and womens data sets into one
    combined output.
    """

    mens_data['League'] = 'M'
    womens_data['League'] = 'W'
    combined = pd.concat([mens_data, womens_data], axis=0)
    return combined

In [6]:
# The input data is structured such that we have one box score per game, with stats identified as the winning and losing team.
# The goal is to create one record for every team per game, with one set of stats columns for that teams game performance, 
# and one set of stat columns for the performance of their opponent.

# Combine Data
mens = pd.read_csv(f'{input_file_path}/MRegularSeasonDetailedResults.csv')
womens = pd.read_csv(f'{input_file_path}/WRegularSeasonDetailedResults.csv')
detailed_results_reg = combine_data(mens, womens)

# Identify key stats columns, along with their Winning/Losing variants for inputs, and the '_against' variant for the output
stat_columns = [
    'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 
    'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF',
]

stat_columns_win = ['W' + col for col in stat_columns]
stat_columns_lose = ['L' + col for col in stat_columns]
stat_columns_against = [col + '_against' for col in stat_columns]
stat_features = stat_columns + stat_columns_against

# Creating maps for stat columns to for depending if we're creating a record for the winning or losing team.
win_to_norm = dict(zip(stat_columns_win, stat_columns))
lose_to_against = dict(zip(stat_columns_lose, stat_columns_against))

lose_to_norm = dict(zip(stat_columns_lose, stat_columns))
win_to_against = dict(zip(stat_columns_win, stat_columns_against))

# Convert H/A/N courts to numeric variable.
# Input data only has H/A/N designation for winning team, so we'll set losing team to the opposite value.
loc_map = {"H":1, "N":0, "A":-1}

detailed_results_reg["win_home_away"] = detailed_results_reg["WLoc"].map(loc_map)
detailed_results_reg["lose_home_away"] = detailed_results_reg["win_home_away"].multiply(-1)

# Create dataset where winning teams are the "target" team, and losing teams are the opponents.
wining_team_stats_reg = detailed_results_reg.rename(
    columns = win_to_norm | lose_to_against | {"WTeamID":"TeamId", "LTeamID":"TeamId_against", "win_home_away":"home_away"}
).drop(['WLoc', 'lose_home_away'], axis=1)
wining_team_stats_reg['Win'] = 1

# Create dataset where losing teams are the "target" team, and winning teams are the opponents.
losing_team_stats_reg = detailed_results_reg.rename(
    columns = lose_to_norm | win_to_against | {"LTeamID":"TeamId", "WTeamID":"TeamId_against", "lose_home_away":"home_away"}
).drop(['WLoc', 'win_home_away'], axis=1)
losing_team_stats_reg['Win'] = 0

# Combine winning and losing sets for final outputs.
team_stats_reg = pd.concat(
    [wining_team_stats_reg, losing_team_stats_reg], 
    ignore_index=True, 
    axis=0
)

team_stats_reg[stat_features] = team_stats_reg[stat_features].astype(float)

team_stats_reg.to_csv(f'{output_file_path}/TeamStatsRegular.csv') 
team_stats_reg

,Season,DayNum,TeamId,Score,TeamId_against,Score_against,NumOT,FGM,FGA,FGM3,...,OR_against,DR_against,Ast_against,TO_against,Stl_against,Blk_against,PF_against,League,home_away,Win
0,2003,10,1104,68.0,1328,62.0,0,27.0,58.0,3.0,...,10.0,22.0,8.0,18.0,9.0,2.0,20.0,M,0,1
1,2003,10,1272,70.0,1393,63.0,0,26.0,62.0,8.0,...,20.0,25.0,7.0,12.0,8.0,6.0,16.0,M,0,1
2,2003,11,1266,73.0,1437,61.0,0,24.0,58.0,8.0,...,31.0,22.0,9.0,12.0,2.0,5.0,23.0,M,0,1
3,2003,11,1296,56.0,1457,50.0,0,18.0,38.0,3.0,...,17.0,20.0,9.0,19.0,4.0,3.0,23.0,M,0,1
4,2003,11,1400,77.0,1208,71.0,0,30.0,61.0,6.0,...,21.0,15.0,12.0,10.0,7.0,1.0,14.0,M,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401175,2025,131,3413,66.0,3471,75.0,0,24.0,67.0,9.0,...,8.0,31.0,10.0,11.0,6.0,1.0,20.0,W,1,0
401176,2025,132,3476,49.0,3192,66.0,0,21.0,57.0,4.0,...,10.0,22.0,11.0,9.0,8.0,1.0,8.0,W,-1,0
401177,2025,132,3119,62.0,3250,74.0,0,25.0,56.0,6.0,...,5.0,25.0,15.0,15.0,6.0,0.0,12.0,W,-1,0
401178,2025,132,3125,62.0,3293,83.0,0,24.0,68.0,2.0,...,5.0,33.0,21.0,13.0,2.0,3.0,15.0,W,0,0


In [10]:
# Along with in-game performances, I am using end of season team rankings to help build the model.

ranks = pd.read_csv(f'{input_file_path}/MMasseyOrdinals.csv')

# Filter to final day, take average  across all available rankings
final_ranks = ranks[ranks["RankingDayNum"]==133]
final_ranks = final_ranks.groupby(['Season', 'TeamID'])["OrdinalRank"].mean().reset_index()
final_ranks = final_ranks.rename({"OrdinalRank":"avg_rank", "TeamID":"TeamId"}, axis=1)

final_ranks.to_csv(f'{output_file_path}/TeamAvgRanks.csv') 
final_ranks

,Season,TeamId,avg_rank
0,2003,1102,156.031250
1,2003,1103,168.000000
2,2003,1104,38.031250
3,2003,1105,308.968750
4,2003,1106,262.687500
...,...,...,...
7622,2025,1476,313.240741
7623,2025,1477,338.788462
7624,2025,1478,347.203704
7625,2025,1479,323.092593


# Feature Engineering

In [11]:
def weight_recent_games(df, stat_columns):
    """
    This method will weight the stat_columns of a given dataset 
    such that games later in the season are weighted more than earlier games.
    The current implentation applies a linear weight.

    :param df: Input game-level data.
    :param stat_columns: List of target stats to weight.
    :return: Game-level data with stats weighted by day of the season (later is higher weight).
    """
    output = df.copy()
    output['Weight'] = 1 + (output['DayNum'] / output.groupby(['League', 'Season'])['DayNum'].transform('max'))

    for col in stat_columns:
        output[col] = output[col] * output['Weight']

    return output

In [12]:
def agg_weight(df, stat_columns):
    """
    This mathod aggregates game data to the season/team level using using a weighted average.

    :param df: Input game-level data.
    :param stat_columns: List of target stats to aggregate.
    :return: Weighted average for each stat at the Season and Team level.
    """
    season_agg = df.groupby(['League', 'Season', 'TeamId']).apply(
        lambda x: (x[stat_columns].sum() / x['Weight'].sum()),
        include_groups=False
    ).reset_index()

    return season_agg

In [ ]:
def normalize_by_opponent(df, stat_columns):
    """
    This method adjust game level stats based on opponent strength. I compare a teams recorded stats to 
    average allowed/achieved by each opponent. 

    Ex. 1 - Normalize target teams Score in a game baed on the opponents average Score_against in a season.
    Ex. 2 - Normalize target temas Ast_against based on the opponents average  Ast in a seaosn.

    :param df: Input game-level data.
    :param stat_columns: List of target stats to normalize.
    :return: Game level stats normalized based on oppoents avg performance for a given season..
    """
    # Get aggregated stats for all teams
    data_agg = agg_weight(df, stat_columns)
    
    # Merge team avaerages to game data
    opp_stats = df.merge(
        data_agg.rename(columns={'TeamId':'TeamId_against'} | {x:f"opp_{x}" for x in stat_columns}),
        on=["League", "Season", "TeamId_against"],
        how='left'
    )

    # Normalize team performance relative to opponent's defense
    for col in stat_columns:
        if "_against" in col:
            opp_stats[col] = opp_stats[col] / opp_stats[f'opp_{col}']
        else:
            opp_stats[col] = opp_stats[col] / opp_stats[f'opp_{col}_against']

    output = opp_stats.drop(columns=[f"opp_{x}" for x in stat_columns])

    return output

In [14]:
def normalize_by_home_court(df, stat_columns):
    """
    This method adjust game level stats based on home/away court performance. Season stats are aggregated by home/away 
    (neutral ignored), and to used to calculate and home court effect per stat. The effect is divided by 2, and then subtracted 
    from home stats and added to away stats to normalize.

    :param df: Input game-level data.
    :param stat_columns: List of target stats to normalize.
    :return: Game level stats normalized based on home/away performance for a given season.
    """
    group_by = ["League", "Season", "TeamId"]
    # Separate aggregates for home and away games
    home_data = df[df['home_away']==1]
    away_data = df[df['home_away']==-1]

    home_data_agg = agg_weight(home_data, stat_columns).rename({stat:stat + "_home" for stat in stat_columns}, axis=1)
    away_data_agg = agg_weight(away_data, stat_columns).rename({stat:stat + "_away" for stat in stat_columns}, axis=1)

    output = df.merge(home_data_agg, on=group_by).merge(away_data_agg, on=group_by)

    # Normalize by home-court advantage
    for stat in stat_columns:
        # Compute home-court performance difference
        effect = (output[f'{stat}_home'] -  output[f'{stat}_away']) / 2  
        output.loc[output['home_away'] == 1, stat] -= effect
        output.loc[output['home_away'] == -1, stat] += effect
        
    output = output[["League", "Season", "TeamId", "DayNum", "Weight"] + stat_columns]
    
    return output

In [ ]:
def scaled_stats(df, group_by, stat_columns, scaler):
    """
    This method scales given stats based on the scaler model provided.

    :param df: Input game-level data.
    :param group_by: List of dimensions over which to scale each state by.
    :param stat_columns: List of target stats to scale.
    :param scaler: Scaling model to use.
    :return: Dataset with scaled stats.
    """
    data_scaled = df.groupby(group_by)[stat_columns].apply(
        lambda x: pd.DataFrame(
            scaler.fit_transform(x),
            columns=stat_columns,
        ),
        include_groups=False
    ).reset_index()
    
    output = df.copy()    
    output[stat_columns] = data_scaled[stat_columns]

    return output

In [18]:
# Select which columns to keep for the analysis, and create '_against' variats.

stat_columns = [
    'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 
    'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF',
]

stat_features = stat_columns + [col + '_against' for col in stat_columns]

team_stats_reg = pd.read_csv(f'{output_file_path}/TeamStatsRegular.csv') 

# The order for stat processing is weighting by day, normalize by opponent, 
# normalize by home_court, and then aggregating to season/team level.
weighted_stats = weight_recent_games(team_stats_reg, stat_features)
normalized_stats = normalize_by_opponent(weighted_stats, stat_features)
normalized_stats = normalize_by_home_court(normalized_stats, stat_features)
normalized_stats_agg = agg_weight(normalized_stats, stat_features) 

normalized_stats_agg.to_csv(f'{output_file_path}/TeamStatsRegularOppHomeNorm.csv') 

# Prep Data For Modeling

In [30]:
def create_matchups(df): 
    """
    This competition creates competition-compliant IDs for each match up, and creates a match-up dataset with
    actual game results that will be used to join final stats on for model training.
    Pred = 1 means the first team_id in the concatenated ID won.

    :param df: Input game-level data.

    :return: Game results with competition compliant ID.
    """
    
    df['TeamID_first'] = df[['WTeamID', 'LTeamID']].min(axis=1)
    df['TeamID_second'] = df[['WTeamID', 'LTeamID']].max(axis=1)
    
    # data[['Season_str', 'TeamID_first_str', 'TeamID_second_str']] = data[['Season', 'TeamID_first', 'TeamID_second']].astype('str')
    df['ID'] = df['Season'].astype('str') + '_' + df['TeamID_first'].astype('str') + '_' + df['TeamID_second'].astype('str') 
   
    df["Pred"] = np.where(df['WTeamID'] < df['LTeamID'], 1, 0)
    
    df = df[["TeamID_first", "TeamID_second", "League", "Season", "ID", "Pred"]]
    
    return df

In [ ]:
def join_matchup_stats(matchup, stats, feature_names):
    """
    This method joins a formatted match dataset with game stats to create a 
    ataset that can be used to train a prediction model.

    :param matchup: Formatted match-up dataframe.
    :param stats: Dataframe containing season and team level stats.
    :param feature_names: List of feature names to use.
    :return: Match-up data with stats used for training.
    """
    combined = matchup.merge(
        stats,
        left_on=["League", 'Season', 'TeamID_first'],
        right_on=["League", 'Season', 'TeamId'],
        how='left'
    ).merge(
        stats,
        left_on=["League", 'Season', 'TeamID_second'],
        right_on=["League", 'Season', 'TeamId'],
        how='left',
        suffixes=('_first', '_second')
    )

    # Features will be expressed as the difference between Team1 features and Team2 features to reduce
    for feat in feature_names:
        combined[feat] = combined[feat+'_first'] - combined[feat+'_second']

    output = combined[["League", "Season", "ID", "Pred"] + feature_names]

    return output

In [21]:
def create_new_stats(df):
    """
    This method takes the aggregated standard stats and calculates higher-level efficiency and rates stats to be used as features.

    :param df: Formatted match-up dataframe.
    :return: Original dataframe with additional efficiency stats.
    """

    output = df.copy()

    # Shooting percentages
    output["FGper"] = output["FGM"]/output["FGA"]
    output["FG3per"] = output["FGM3"]/output["FGA3"]
    output["FTper"] = output["FTM"]/output["FTA"]

    output["FGper_against"] = output["FGM_against"]/output["FGA_against"]
    output["FG3per_against"] = output["FGM3_against"]/output["FGA3_against"]
    output["FTper_against"] = output["FTM_against"]/output["FTA_against"]

    # Total number of possessions
    output["Possessions"] = output["FGA"] + 0.44*output["FTA"] - output["OR"] + output["TO"]
    output["Possessions_against"] = output["FGA_against"] + 0.44*output["FTA_against"] - output["OR_against"] + output["TO_against"]

    # Offensive and Defensive Efficiency
    output["OEFF"] = output["Score"]/output["Possessions"]
    output["DEFF"] = output["Score_against"]/output["Possessions_against"]
    output["NET_EFF"] = output["OEFF"] - output["DEFF"]
    
    # Shooting Efficiency
    output["eFG"] = (output["FGM"] + 0.5 * output["FGM3"]) / output["FGA"]
    output["TS"] = output["Score"] / (2 * (output["FGA"] + 0.44 * output["FTA"]))

    # Rebounding Percentages
    output["ORper"] = output["OR"] / (output["OR"] + output["DR_against"])
    output["DRper"] = output["DR"] / (output["DR"] + output["OR_against"]) 
    
    # Turnover and Assist Ratios
    output["TOper"] = output["TO"] / output["Possessions"]
    output["AST_TO"] = output["Ast"] / output["TO"]

    # 3-Point Reliance
    output["3P_Reliance"] = output["FGA3"] / output["FGA"]

    # Free Throw Rate
    output["FTR"] = output["FTA"] / output["FGA"]
    
    # Defensive Stats
    output["STLper"] = output["Stl"] / output["Possessions"]
    output["BLKper"] = output["Blk"] / output["FGA"]
    
    return output

In [24]:
# Select initial features to keep
stat_columns = [
    'Score', 'FGper', 'FG3per', 'FTper', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 
]

stat_features = stat_columns + [col + '_against' for col in stat_columns]

# Select efficency/rate features to keep
eff_metrics =  ["OEFF", "DEFF", "NET_EFF", "eFG", "TS", "ORper", "DRper", "TOper", "AST_TO", "3P_Reliance", "FTR", "STLper", "BLKper"]

# Combine regular and efficiency features, with avg rank, which comes from a whole separate set
all_features = stat_features + eff_metrics + ["avg_rank"]

stats = pd.read_csv(f'{output_file_path}/TeamStatsRegularOppHomeNorm.csv')
stats = create_new_stats(stats)

ranks = pd.read_csv(f'{output_file_path}/TeamAvgRanks.csv') 

# Scale season stats with Standard Scaler
scaler = StandardScaler()  
stats_scaled = scaled_stats(stats, ['League', 'Season'], stat_features, scaler).fillna(0)

# Scale average ranks Min/Max Scaler
scaler = MinMaxScaler()
ranks_scaled = scaled_stats(final_ranks, ['Season'], ["avg_rank"], scaler).fillna(0)

# Combined Stats and Ranks to create final feature set
combined_data = stats_scaled.merge(ranks_scaled, on=["Season", "TeamId"], how='left')
combined_data.to_csv(f'{output_file_path}/CombinedSeasonStats.csv') 
combined_data

,Unnamed: 0,League,Season,TeamId,Score,FGM,FGA,FGM3,FGA3,FTM,...,TS,ORper,DRper,TOper,AST_TO,3P_Reliance,FTR,STLper,BLKper,avg_rank
0,0,M,2003,1102,-2.500885,0.774987,0.732575,1.185559,1.174275,0.739722,...,0.372136,0.295262,0.461696,0.531544,1.165423,1.602942,1.077354,0.596442,0.710962,0.476089
1,1,M,2003,1103,1.218835,1.081192,1.004299,0.889303,0.922156,1.195015,...,0.363129,0.463639,0.450828,0.576941,1.271202,0.918209,1.130546,0.671191,0.861563,0.512907
2,2,M,2003,1104,0.344718,1.004132,1.030245,1.026952,1.091218,1.113105,...,0.340558,0.535242,0.530517,0.743949,1.028743,1.059184,1.045126,0.743991,1.108544,0.113100
3,3,M,2003,1105,0.055105,0.969069,1.057089,1.222345,1.118141,1.035631,...,0.335390,0.489881,0.487977,0.715033,0.915044,1.057755,0.938700,0.704807,0.658021,0.946551
4,4,M,2003,1106,-1.493455,0.911095,0.938470,0.941342,0.936593,0.743614,...,0.345271,0.510200,0.524002,0.764057,0.756047,0.998000,0.819873,0.773336,1.093495,0.804182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13535,13535,W,2025,3476,-0.053639,1.004894,1.004298,1.171706,1.132094,0.826031,...,0.373989,0.523157,0.501541,0.782764,1.098133,1.127250,0.727098,0.433582,0.938527,NaN
13536,13536,W,2025,3477,-0.462800,0.987985,1.024610,1.198481,1.140033,0.695858,...,0.357825,0.367421,0.450640,0.596812,1.126157,1.112650,0.672728,0.479949,1.121814,NaN
13537,13537,W,2025,3478,-1.364512,0.790333,0.904093,1.027236,1.019455,1.044451,...,0.320852,0.438334,0.462034,0.675454,0.714581,1.127600,1.094270,0.458743,0.687772,NaN
13538,13538,W,2025,3479,-0.147864,0.915304,0.914897,1.040519,1.060513,1.232333,...,0.336323,0.359508,0.527391,0.590825,0.613360,1.159162,1.353979,0.359670,0.438956,NaN


In [ ]:
# Use Tournament Game results to create match up data that will be used to test potential models.

mens = pd.read_csv(f'{input_file_path}/MNCAATourneyCompactResults.csv')
womens = pd.read_csv(f'{input_file_path}/WNCAATourneyCompactResults.csv')

tourney_results = combine_data(mens, womens)
tourney_results = tourney_results[(tourney_results["Season"]>=2003) & (tourney_results["Season"]<=2024)]
tourney_matchups = create_matchups(tourney_results)

tourney_data = join_matchup_stats(tourney_matchups, combined_data, all_features)
tourney_data.to_csv(f'{output_file_path}/TournamentDataModel.csv') 
tourney_data

,League,Season,ID,Pred,Score,FGper,FG3per,FTper,OR,DR,...,TS,ORper,DRper,TOper,AST_TO,3P_Reliance,FTR,STLper,BLKper,avg_rank
0,M,2003,2003_1411_1421,0,-0.428417,0.534466,-1.171753,-3.631951,-0.288813,-0.340678,...,-0.016673,0.014286,0.056440,-0.051792,0.074402,0.058593,0.294650,-0.045537,-0.040945,-0.003268
1,M,2003,2003_1112_1436,1,2.617211,0.853587,0.201331,1.274569,1.036149,1.433230,...,0.015377,0.004707,-0.052686,-0.042102,0.222687,0.129852,0.104858,0.152532,0.308251,-0.462806
2,M,2003,2003_1113_1272,1,-0.171535,1.290227,-1.415006,0.219078,0.077096,-1.398818,...,-0.006731,0.032263,-0.001898,0.005906,-0.096973,-0.317014,0.252727,-0.187017,-0.082094,0.043971
3,M,2003,2003_1141_1166,1,-0.055723,0.317801,0.572552,2.120162,-0.362059,0.177113,...,0.003098,0.023491,-0.019497,0.043902,-0.368421,-0.057083,0.223405,-0.288828,0.125715,0.076757
4,M,2003,2003_1143_1301,1,0.081432,-0.075555,1.003634,-2.557847,1.191003,0.634562,...,-0.008879,0.040040,-0.012052,0.044130,0.125341,-0.282150,-0.156112,0.050039,-0.032528,-0.042778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2712,W,2024,2024_3163_3425,1,1.261665,2.612261,0.719230,0.184715,-1.567879,0.758415,...,0.058515,-0.026601,0.031223,-0.087392,0.443065,-0.049623,-0.161565,0.032652,-0.418231,NaN
2713,W,2024,2024_3234_3261,1,0.924604,0.614796,1.166128,0.306344,-2.548397,0.796243,...,0.048311,-0.119363,-0.022279,-0.212875,0.427734,0.699716,-0.185788,-0.459657,-0.346681,NaN
2714,W,2024,2024_3163_3234,0,-1.018758,0.439505,-0.294310,-0.347323,-0.738121,-0.868309,...,-0.003486,-0.013028,0.040343,-0.023255,0.007759,-0.358853,-0.072845,0.261253,0.128289,NaN
2715,W,2024,2024_3301_3376,0,-1.735155,-2.256180,-2.225698,0.862753,-1.507308,0.049544,...,-0.037720,-0.117567,0.009465,-0.116196,-0.316944,0.122649,-0.068777,-0.279660,-0.791674,NaN


In [ ]:
# Use Regular Season Game results to create match up data that will be used to train models.

mens = pd.read_csv(f'{input_file_path}/MRegularSeasonCompactResults.csv')
womens = pd.read_csv(f'{input_file_path}/WRegularSeasonCompactResults.csv')

rs_results = combine_data(mens, womens)
rs_results = rs_results[(rs_results["Season"]>=2003) & (rs_results["Season"]<=2024)]
rs_matchups = create_matchups(rs_results)

rs_data = join_matchup_stats(rs_matchups, combined_data, all_features)
rs_data.to_csv(f'{output_file_path}/RegularDataModel.csv') 
rs_data

,League,Season,ID,Pred,Score,FGper,FG3per,FTper,OR,DR,...,TS,ORper,DRper,TOper,AST_TO,3P_Reliance,FTR,STLper,BLKper,avg_rank
0,M,2003,2003_1104_1328,1,-0.000740,-1.330504,-2.856452,0.198372,1.621103,0.054283,...,-0.030199,0.033948,-0.018543,0.073535,-0.222358,-0.000393,0.070003,-0.043531,0.004040,0.098172
1,M,2003,2003_1272_1393,1,-0.800060,-1.247011,1.629973,-0.801230,-0.055437,-0.555612,...,-0.009728,-0.008317,0.056451,0.021243,0.136634,0.219711,0.011817,-0.065957,-0.292904,0.026509
2,M,2003,2003_1266_1437,1,1.173452,2.142265,1.124147,1.671130,-0.075563,-0.135540,...,0.025453,0.059489,-0.046505,-0.037794,0.358097,-0.180591,0.179605,-0.093667,0.243783,-0.189439
3,M,2003,2003_1296_1457,1,-0.095791,0.770340,0.904515,0.625037,1.046005,-0.534589,...,0.010475,0.070036,-0.001157,0.132550,-0.154666,-0.118495,0.043036,0.128498,-0.238387,-0.224080
4,M,2003,2003_1208_1400,0,-0.039576,0.347137,0.461716,-0.508841,-1.472740,-0.003026,...,0.010251,-0.077918,-0.017221,-0.122227,0.551923,0.010359,-0.148898,0.113797,0.173833,0.016798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223433,W,2024,2024_3372_3465,0,-1.053799,0.139839,-0.649752,0.193688,0.569588,-1.715288,...,-0.011554,0.064852,-0.024112,0.098382,-0.371861,-0.271696,0.052872,0.044822,0.732491,NaN
223434,W,2024,2024_3179_3283,1,1.729851,1.612591,0.041896,1.588058,-1.180120,2.709310,...,0.050465,-0.041240,0.000181,-0.094055,0.101986,0.500608,0.019208,-0.263945,0.454934,NaN
223435,W,2024,2024_3180_3392,1,-2.371720,-0.655695,0.016563,-0.407146,-0.988930,-3.591421,...,-0.008643,-0.050713,-0.052416,0.011259,0.381588,-0.275599,-0.297633,0.114952,-0.348851,NaN
223436,W,2024,2024_3131_3221,0,0.010028,1.010914,0.616310,-1.005953,-1.768765,-0.507977,...,0.015763,-0.056815,-0.103827,-0.069990,-0.238978,0.113769,0.153570,0.038567,0.668338,NaN
